# NHANES Diastolic Blood Pressure Analysis

This notebook presents a comprehensive statistical analysis of diastolic blood pressure using data from the **National Health and Nutrition Examination Survey (NHANES) 2015-2016**. The primary objective is to explore how dependent data structures—specifically geographic and demographic clustering—influence regression outcomes.

The analysis utilizes two advanced statistical frameworks to account for the survey's complex design:

+ **Generalized Estimating Equations (GEE)**: Used to fit marginal linear models that account for correlations within clusters.

+ **Multilevel (Mixed-Effects) Models**: Used to estimate both fixed effects of covariates and random intercepts for grouping variables.

In [1]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf

print("Libraries imported successfully")

Libraries imported successfully


To get started, we will use the following code to import the data, filter required columns, handle missing values, and create the Masked Variance Unit `MVU`.

In [ ]:
# Read the data
df = pd.read_csv("nhanes_2015_2016.csv")

# Define columns to keep
keep_cols = [
    "BPXSY1", "BPXDI1", "RIDAGEYR", "RIAGENDR", 
    "RIDRETH1", "DMDEDUC2", "BMXBMI", "SMQ020", 
    "SDMVSTRA", "SDMVPSU"
]

# Keep selected columns and drop rows with any missing values
df_clean = df[keep_cols].dropna().copy()

# Create a unique Masked Variance Unit (MVU) by combining Stratum and PSU
# This is used for cluster-based analysis
df_clean['MVU'] = df_clean['SDMVSTRA'].astype(str) + "-" + df_clean['SDMVPSU'].astype(str)

# Save the processed data
df_clean.to_csv('nhanes_processed.csv', index=False)

print(f"Final dataset shape: {df_clean.shape}")
print(f"\nNumber of unique MVU groups: {df_clean['MVU'].nunique()}")
print(f"\n----- Distribution of the observations per MVU -----")
print(df_clean['MVU'].value_counts().describe())


Final dataset shape: (5102, 11)

Number of unique MVU groups: 30

----- Distribution of the observations per MVU -----
count     30.000000
mean     170.066667
std       30.073397
min      106.000000
25%      155.250000
50%      172.000000
75%      183.500000
max      226.000000
Name: count, dtype: float64


### Data Dictionary

The following table describes the variables included in the processed dataset:

| Column    | Description                  | Values / Units                                                                                                   |
|----------|------------------------------|------------------------------------------------------------------------------------------------------------------|
| BPXSY1   | Systolic Blood Pressure      | mmHg (1st reading)                                                                                               |
| BPXDI1   | Diastolic Blood Pressure     | mmHg (1st reading)                                                                                               |
| RIDAGEYR | Age at Screening             | Years (0 to 80+, where 80 is top-coded)                                                                          |
| RIAGENDR | Gender                       | 1: Male, 2: Female                                                                                               |
| RIDRETH1 | Race/Hispanic Origin         | 1: Mexican American, 2: Other Hispanic, 3: Non-Hispanic White, 4: Non-Hispanic Black, 5: Other Race/Multi-Racial |
| DMDEDUC2 | Education Level (Adults 20+) | 1: < 9th Grade, 2: 9-11th, 3: HS Grad, 4: Some College, 5: College Grad                                          |
| BMXBMI   | Body Mass Index              | kg/$m^2$                                                                                                            |
| SMQ020   | Smoked 100+ Cigarettes       | 1: Yes, 2: No                                                                                                    |
| SDMVSTRA | Masked Variance Stratum      | Sample design variable for variance estimation                                                                   |
| SDMVPSU  | Masked Variance PSU          | Primary Sampling Unit                                                                                            |
| MVU      | Masked Variance Unit         | Combined Stratum and PSU (e.g., "125_1")                                                                         |

### Build a marginal linear model for the first measurement of diastolic blood pressure `BPXDI1`, using GEE to account for the grouping variable 'group' defined above. This initial model should have no covariates, and will be used to assess the ICC of this blood pressure measure. 

To build the marginal linear model for the first measurement of diastolic blood pressure `BPXDI1`, we use Generalized Estimating Equations (GEE) with an exchangeable correlation structure. This structure assumes that every pair of observations within the same cluster (group) has the same correlation (α), which corresponds to the Intra-class Correlation Coefficient (ICC).

In this context, we use the MVU (Masked Variance Unit) created earlier as the grouping variable.

First, we need to import the necessary libraries for GEE analysis.

In [14]:
# Ensure the data is sorted by the grouping variable
df_clean = df_clean.sort_values(by='MVU')

# Fit the GEE model with no covariates
# Using the Gaussian family for a linear model
gee_model = smf.gee("BPXDI1 ~ 1",
                    groups='MVU',
                    data=df_clean,
                    family=sm.families.Gaussian(),
                    cov_struct=sm.cov_struct.Exchangeable())

gee_results = gee_model.fit()

# Display the summary and the estimated correlation parameter (ICC)
print(gee_results.summary())
print(gee_results.cov_struct.summary())


                               GEE Regression Results                              
Dep. Variable:                      BPXDI1   No. Observations:                 5102
Model:                                 GEE   No. clusters:                       30
Method:                        Generalized   Min. cluster size:                 106
                      Estimating Equations   Max. cluster size:                 226
Family:                           Gaussian   Mean cluster size:               170.1
Dependence structure:         Exchangeable   Num. iterations:                     5
Date:                     Sat, 14 Mar 2026   Scale:                         162.315
Covariance type:                    robust   Time:                         00:42:24
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     70.0088      0.419    167.219      0.000      69.188      70.829
Skew:  

#### Key Findings:

+ **Mean Diastolic BP**: The intercept estimate of 70.01 represents the average diastolic blood pressure across the sample, accounting for the clustering.

+ **Intra-class Correlation Coefficient (ICC)**: The estimated correlation between two observations in the same MVU is 0.031.

+ **Interpretation**: This value indicates that approximately 3.1% of the total variation in diastolic blood pressure is attributable to differences between the variance units, while the remaining 96.9% is due to within-group variation or other individual factors. A low ICC suggests that the grouping by MVU (stratum and PSU) does not account for a large portion of the variance in blood pressure, but it is still non-zero, justifying the use of cluster-robust methods.

###  What can you conclude by comparing the ICC for diastolic blood pressure to the ICC for systolic blood pressure?

To compare the Intra-class Correlation Coefficients (ICC) for Systolic Blood Pressure (SBP) and Diastolic Blood Pressure (DBP), we fitted identical marginal models using GEE with an exchangeable correlation structure. The results are as follows:

In [15]:
# Ensure the data is sorted by the grouping variable
df_clean = df_clean.sort_values(by='MVU')

# Fit GEE for Systolic Blood Pressure (BPXSY1)
model_sbp = smf.gee("BPXSY1 ~ 1", groups='MVU', data=df_clean,
                    family=sm.families.Gaussian(),
                    cov_struct=sm.cov_struct.Exchangeable())

result_sbp = model_sbp.fit()

# Fit GEE for Diastolic Blood Pressure (BPXDI1) again to bee sure of the
# comparison in the same context
model_dbp = smf.gee("BPXDI1 ~ 1", groups="MVU", data=df_clean,
                    family=sm.families.Gaussian(),
                    cov_struct=sm.cov_struct.Exchangeable())
result_dbp = model_dbp.fit()

print(f"Systolic Blood Pressure (SBP) ICC: {result_sbp.cov_struct.summary()}")
print(f"Diasstolic Blood Pressure (DBP) ICC: {result_dbp.cov_struct.summary()}")

# Also check variances (scale parameter) to see total variability
print(f"SBP Scale (Variance): {result_sbp.scale}")
print(f"DBP Scale (Variance): {result_dbp.scale}")

Systolic Blood Pressure (SBP) ICC: The correlation between two observations in the same cluster is 0.030
Diasstolic Blood Pressure (DBP) ICC: The correlation between two observations in the same cluster is 0.031
SBP Scale (Variance): 341.8379439598926
DBP Scale (Variance): 162.31541101752006


#### Conclusions from the Comparison

1. **Similirity in Clustering Impact**

+ The ICC values for both SBP (3.0%) and DBP (3.1%) are remarkably similar. This suggests that the geographic and demographic clustering represented by the Masked Variance Units (MVU) affects both components of blood pressure in a nearly identical relative proportion. In both cases, only about 3% of the total variation is due to differences between the variance units, while the vast majority (approx. 97%) is due to individual-level differences.


2. **Comparison of Total Variability**

+ While the ICCs are almost the same, the absolute variability is significantly different. The scale parameter (variance) for SBP (341.84) is more than twice as large as that for DBP (162.32).

    * This indicates that although SBP is much more "spread out" and variable than DBP, the proportion of that spread that is explained by the survey clusters remains consistent with DBP.

3. **Biological Consistency**

+ From a biological perspective, this suggests that the environmental or regional factors that might influence blood pressure (such as local diet trends, altitude, or regional socioeconomic status represented by the MVU) do not affect one type of pressure over the other. They appear to shift the entire blood pressure profile (both systolic and diastolic) by a similar relative amount.

### Let's take our `gee_model`, and add sex, age, and BMI  as covariates.

To expand our analysis, we have updated the marginal linear model for diastolic blood pressure `BPXDI1` to include sex `RIAGENDR`, age `RIDAGEYR`, and BMI `BMXBMI` as covariates. We continue to use the Generalized Estimating Equations (GEE) framework with an exchangeable correlation structure to account for the clustering within the Masked Variance Units (MVU).

In [16]:
# Ensure the data is sorted by the grouping variable
df_clean = df_clean.sort_values(by='MVU')

# Define the model formula with covariates
# RIAGENDR is categorical (1=Male, 2=Female)
# RIDAGEYR and BMXBMI are continuous
formula = "BPXDI1 ~ C(RIAGENDR) + RIDAGEYR + BMXBMI"

# Fit the GEE model
model_cov = smf.gee(formula, groups="MVU", data=df_clean,
                    family=sm.families.Gaussian(),
                    cov_struct=sm.cov_struct.Exchangeable())

result_cov = model_cov.fit()

# Print the results
print(result_cov.summary())
print(f"\nEstimated ICC with covariates:")
print(result_cov.cov_struct.summary())

# Compare with intercept-only model scale (variance)
print(f"\nResidual Variance (Scale): {result_cov.scale}")

                               GEE Regression Results                              
Dep. Variable:                      BPXDI1   No. Observations:                 5102
Model:                                 GEE   No. clusters:                       30
Method:                        Generalized   Min. cluster size:                 106
                      Estimating Equations   Max. cluster size:                 226
Family:                           Gaussian   Mean cluster size:               170.1
Dependence structure:         Exchangeable   Num. iterations:                     7
Date:                     Sat, 14 Mar 2026   Scale:                         158.611
Covariance type:                    robust   Time:                         01:30:02
                       coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------
Intercept           68.0436      1.097     62.004      0.000      65.893  

#### Interpretation of Covariates

+ **Sex `RIAGENDR`**: On average, females (coded as 2) have a diastolic blood pressure that is 2.74 mmHg lower than males, holding age and BMI constant. This difference is highly statistically significant (p<0.001).

+ **Age `RIDAGEYR`**: For every one-year increase in age, diastolic blood pressure is expected to decrease by 0.04 mmHg. While the effect size is small, it is statistically significant (p=0.004).

+ **Body Mass Index `BMXBMI`**: For every 1 kg/$m^2$ increase in BMI, diastolic blood pressure increases by 0.18 mmHg. This positive relationship is highly significant (p<0.001).


#### Impact on the Intra-class Correlation Coefficient (ICC)

+ **New ICC (with covariates)**: 0.030

+ **Previous ICC (No covariates)**: 0.031

**Observations**:
The ICC remains nearly unchanged (0.030 vs. 0.031) after adding sex, age, and BMI to the model. This indicates that the similarity between individuals in the same `MVU` is not explained by them having similar ages, genders, or BMI profiles; the "cluster effect" likely stems from other unmeasured environmental or regional factors.

### Split the data into separate datasets for females and for males and fit two separate marginal linear models for diastolic blood pressure, one only for females, and one only for males.

In [19]:
# Split into male and female datasets
df_male = df_clean[df_clean['RIAGENDR'] == 1].copy().sort_values(by='MVU')
df_female = df_clean[df_clean['RIAGENDR'] == 2].copy().sort_values(by='MVU')

# Define model formula excluding sex since we are splitting
formula = "BPXDI1 ~ RIDAGEYR + BMXBMI"

# Fit Gee for Males
model_male = smf.gee(formula, groups='MVU', data=df_male,
                     family=sm.families.Gaussian(),
                     cov_struct=sm.cov_struct.Exchangeable())
result_male = model_male.fit()

# Fit GEE for Females
model_female = smf.gee(formula, groups='MVU', data=df_female,
                     family=sm.families.Gaussian(),
                     cov_struct=sm.cov_struct.Exchangeable())
result_female = model_female.fit()

# Prepare summaries
print("--- Male Model Results ---")
print(result_male.summary())
print(f"Male ICC: {result_male.cov_struct.summary()}")

print("\n\n--- Female Model Results ---")
print(result_female.summary())
print(f"Female ICC: {result_female.cov_struct.summary()}")

--- Male Model Results ---
                               GEE Regression Results                              
Dep. Variable:                      BPXDI1   No. Observations:                 2462
Model:                                 GEE   No. clusters:                       30
Method:                        Generalized   Min. cluster size:                  38
                      Estimating Equations   Max. cluster size:                 111
Family:                           Gaussian   Mean cluster size:                82.1
Dependence structure:         Exchangeable   Num. iterations:                     7
Date:                     Sat, 14 Mar 2026   Scale:                         173.641
Covariance type:                    robust   Time:                         03:09:53
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     66.8949      1.751     38.210      0.000     

We have split the data into two separate datasets and fitted marginal linear models for diastolic blood pressure `BPXDI1` for males and females independently, using age `RIDAGEYR` and BMI `BMXBMI` as covariates.

#### Key Observations and Conclusions

1. **Differences in Covariate Impact**

+ **Age**: Age has a significant negative association with diastolic blood pressure in males (β=−0.066,p<0.001), meaning older men in this sample tend to have lower diastolic BP. However, for females, the effect of age is much smaller and not statistically significant (p=0.318).

+ **BMI**: BMI is a significant predictor for both sexes, but the effect size is twice as large in males (β=0.265) compared to females (β=0.132). This suggests that an increase in BMI is associated with a steeper rise in diastolic blood pressure for men than for women.

2. **Variability and Clustering**

+ **Variance**: Diastolic blood pressure is more variable in males (Scale = 173.64) than in females (Scale = 143.98).

+ **ICC**: The ICC is slightly higher for males (0.035) than for females (0.029). While both are low, this indicates that male diastolic blood pressure may be slightly more sensitive to the geographic/environmental factors represented by the MVU clusters than female diastolic blood pressure.

3. **Interpretation of Results**

+ By splitting the models, we can see that the relationship between age, BMI.and diastolic blood pressure is not uniform across genders. The combined model previously showed a general trend, but the stratified models reveal that the age-related decline in diastolic BP and the impact of BMI are notably more pronounced in the male population of this dataset.

### Using the entire data set, fit a multilevel model for diastolic blood pressure, predicted by age, gender, BMI, and educational attainment. Include a random intercept for groups.

To analyze the diastolic blood pressure `BPXDI1` across the entire dataset while accounting for the group-level clustering, we fitted a multilevel (mixed-effects) model. This model includes a random intercept for each variance unit (group) and treats age, gender, BMI, and educational attainment as fixed effects.

In [22]:
# Ensure the grouping variable 'group' is present
df_clean['group'] = df_clean['MVU']

# Define the multilevel model (Mixed Linear Model)
formula = "BPXDI1 ~ RIDAGEYR + C(RIAGENDR) + BMXBMI + C(DMDEDUC2)"

# Fit the model using MixedLM for random intercepts
model_lmm = smf.mixedlm(formula, df_clean, groups=df_clean['group'])
result_lmm = model_lmm.fit()

# Print the summary
print(result_lmm.summary())

# Calculate the ICC from the Mixed Model components
var_resid = result_lmm.scale
var_intercept = result_lmm.cov_re.iloc[0, 0]
icc_lmm = var_intercept / (var_intercept + var_resid)

print(f"\nEstimated Variance of Random Intercept: {var_intercept:.4f}")
print(f"Residual Variance: {var_resid:.4f}")
print(f"Calculated ICC: {icc_lmm:.4f}")

            Mixed Linear Model Regression Results
Model:               MixedLM  Dependent Variable:  BPXDI1     
No. Observations:    5102     Method:              REML       
No. Groups:          30       Scale:               154.2055   
Min. group size:     106      Log-Likelihood:      -20115.7588
Max. group size:     226      Converged:           Yes        
Mean group size:     170.1                                    
--------------------------------------------------------------
                   Coef.  Std.Err.   z    P>|z|  [0.025 0.975]
--------------------------------------------------------------
Intercept          67.445    1.142 59.082 0.000  65.207 69.682
C(RIAGENDR)[T.2]   -2.756    0.351 -7.860 0.000  -3.444 -2.069
C(DMDEDUC2)[T.2.0]  0.736    0.726  1.013 0.311  -0.688  2.159
C(DMDEDUC2)[T.3.0] -0.139    0.644 -0.216 0.829  -1.401  1.122
C(DMDEDUC2)[T.4.0]  0.527    0.618  0.852 0.394  -0.685  1.739
C(DMDEDUC2)[T.5.0]  0.955    0.641  1.489 0.137  -0.302  2.212
C(DMD

#### Key Findings and Conclusions

+ **Covariate Significance**:

    + Sex, Age, and BMI remain highly significant predictors of diastolic blood pressure, consistent with our previous GEE models.

    + Interestingly, education level does not show a statistically significant relationship with diastolic blood pressure in this model (p>0.05 for all categories), suggesting that once age, sex, and BMI are accounted for, education is not a strong independent driver of diastolic BP in this population.

+ **Clustering Influence**:

    + The ICC from the multilevel model is 2.6%. This is slightly lower than the ICC from the intercept-only GEE model (3.1%), indicating that some of the initial group-level variation was explained by the addition of individual-level covariates.

    + The *Group Var* of 4.17 represents the variance in the average diastolic blood pressure between the different Masked Variance Units.

+ **GEE vs. Multilevel Model**:

    + The coefficients for Sex, Age, and BMI in this multilevel model are very similar to those obtained in the GEE model. This is expected in a linear model with a large sample size, as the two methods generally produce similar point estimates but different interpretations of the variance structure. While GEE focuses on the population-averaged effect, this multilevel model allows us to explicitly quantify the variance between groups.

### Using the entire data set, fit a multilevel model for diastolic blood pressure, predicted by age, gender, BMI, and educational attainment. Include a random intercept for age.

To fulfill this request, we have modified the multilevel model to use age `RIDAGEYR` as the grouping variable for the random intercept. This models the variation in diastolic blood pressure across different age groups, while still including age as a fixed effect to capture the overall linear trend.

In [24]:
# Define the multilevel model
formula = "BPXDI1 ~ RIDAGEYR + C(RIAGENDR) + BMXBMI + C(DMDEDUC2)"

# Fit the model with age as the grouping variable
model_age_random = smf.mixedlm(formula, df_clean, groups=df_clean['RIDAGEYR'])
result_age_random = model_age_random.fit()

# Print the summary
print(result_age_random.summary())

# Variance components
var_resid = result_age_random.scale
var_intercept = result_age_random.cov_re.iloc[0, 0]
icc_age = var_intercept / (var_intercept + var_resid)

print(f"\nEstimated Variance of Random Intercept (Age Groups): {var_intercept:.4f}")
print(f"Residual Variance: {var_resid:.4f}")
print(f"Calculated ICC: {icc_age:.4f}")

            Mixed Linear Model Regression Results
Model:               MixedLM  Dependent Variable:  BPXDI1     
No. Observations:    5102     Method:              REML       
No. Groups:          61       Scale:               141.9737   
Min. group size:     33       Log-Likelihood:      -19950.2294
Max. group size:     321      Converged:           Yes        
Mean group size:     83.6                                     
--------------------------------------------------------------
                   Coef.  Std.Err.   z    P>|z|  [0.025 0.975]
--------------------------------------------------------------
Intercept          69.875    1.909 36.606 0.000  66.133 73.616
C(RIAGENDR)[T.2]   -2.755    0.338 -8.144 0.000  -3.418 -2.092
C(DMDEDUC2)[T.2.0]  0.458    0.693  0.661 0.509  -0.901  1.817
C(DMDEDUC2)[T.3.0] -0.042    0.609 -0.069 0.945  -1.236  1.152
C(DMDEDUC2)[T.4.0]  0.786    0.583  1.349 0.177  -0.356  1.928
C(DMDEDUC2)[T.5.0]  0.817    0.600  1.361 0.173  -0.359  1.994
C(DMD

#### Key Observations and Comparison

+ **High Clustering by Age**: The ICC for age groups (10.8%) is significantly higher than the ICC we previously found for the variance units (2.6%). This suggests that individuals of the same age are more similar to each other in terms of diastolic blood pressure than individuals in the same geographic/survey cluster.

+ **Shift in Significance for Age (Fixed Effect)**:
    In this model, the fixed effect of age is no longer statistically significant (p=0.172). This is because the random intercept for age is now capturing the group-to-group differences. 

+ **BMI and Sex**:
    Even with age treated as a random effect, Sex and BMI remain highly significant predictors. This reinforces the idea that biological sex and body mass have consistent, measurable impacts on diastolic blood pressure across all ages.

+ **Education**:
    As with the previous model, educational attainment does not show a significant independent association with diastolic blood pressure when controlling for the other factors.

This model effectively demonstrates that there is substantial "age-group" variation in blood pressure that a simple linear trend might not fully capture.